In [ ]:
import pandas as pd

df1 = pd.read_csv("Datasets/open-meteo.csv")
df2 = pd.read_csv("Datasets/hourly_power_error.csv")

In [840]:
for idx, i in enumerate(df1["time"].apply(lambda x: x.split("T"))):
    df1.loc[idx, "Time"] = i[1]
    df1.loc[idx, "Date"] = i[0]

In [841]:
hours = df1["Time"].str.split(":").str[0].astype(int)
mask = (hours >= 7) & (hours <= 18)

In [842]:
df1["hour"] = df1["Time"].str.split(":").str[0].astype(int)

In [843]:
filtered_df = df1[mask].reset_index(drop=True)

In [844]:
df3 = filtered_df[[col for col in df1.columns if col not in ["Date", "Time", "time"]]]

In [845]:
df2["error"] = df2["error_pct"] / 100

In [846]:
new_df = pd.concat(
    [
        df2[
            [
                col
                for col in df2.columns
                if col
                not in ["date", "hour", "error_pct", "power_kw", "plant_capacity_kw"]
            ]
        ],
        df3,
    ],
    axis=1,
)

In [847]:
from my_tools import outliers_removal

new_df = outliers_removal("error", new_df)

In [848]:
X, y = new_df[[col for col in new_df if col != "error"]], new_df["error"]

In [849]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [850]:
new_df

,error,wind_speed_10m (m/s),pressure_msl (hPa),relative_humidity_2m (%),cloud_cover (%),wind_direction_10m (°),temperature_2m (°C),hour
0,0.8833,4.09,1003.6,70,0,102,27.0,7
1,0.7083,3.52,1003.9,65,0,113,28.8,8
2,0.5200,3.65,1003.8,58,0,123,30.9,9
3,0.3567,3.36,1003.9,53,0,132,32.8,10
4,0.2500,3.12,1004.1,50,0,125,33.5,11
...,...,...,...,...,...,...,...,...
523,0.9033,3.80,999.5,58,48,140,31.5,14
524,0.1000,3.79,999.1,58,44,139,31.6,15
525,0.6833,3.52,998.6,58,33,141,31.6,16
526,0.7083,3.22,999.0,61,24,138,30.9,17


In [851]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
import numpy as np

mm_scale = MinMaxScaler()
ss = StandardScaler()
sdg_reg = SGDRegressor(
    penalty="l2",
    alpha=0.01,
    max_iter=100,
    eta0=0.01615,
    learning_rate="invscaling",
    random_state=40,
)

X_scaled = ss.fit_transform(X_train)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_train)
X_scaled = ss.fit_transform(X_poly)
ss2 = StandardScaler()
X_poly_scaled = ss2.fit_transform(X_poly)


from sklearn.metrics import r2_score

lin_reg = LinearRegression()
lin_reg.fit(X_scaled, y_train)
sdg_reg.fit(X_scaled, y_train)

y_pred = lin_reg.predict(X_scaled)
y_pred_test = lin_reg.predict((ss.transform(poly.transform(X_test))))
r2_score(y_train, y_pred), r2_score(y_test, y_pred_test)

(0.7401095791318018, 0.6395326523457233)

In [852]:
from sklearn.tree import DecisionTreeRegressor

dec_tree = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=3,
    max_features=10,
    random_state=11,
)
dec_tree.fit(X_train, y_train)
y_pred = dec_tree.predict(X_train)
r2_score(y_train, y_pred)
y_pred_test = dec_tree.predict((X_test))
r2_score(y_train, y_pred), r2_score(y_test, y_pred_test)

(0.580604007097623, 0.5768102709963283)

In [868]:
from sklearn.svm import SVR

svr = SVR(kernel="rbf", gamma=0.01, coef0=0.6, C=0.9)
svr.fit(X_scaled, y_train)
y_pred = svr.predict(X_scaled)
y_pred_test = svr.predict(ss.transform(poly.transform(X_test)))
r2_score(y_train, y_pred), r2_score(y_test, y_pred_test)

(0.737790117764214, 0.65162435116872)

In [869]:
import joblib

joblib.dump(svr, "solar_power_error.pkl")

['solar_power_error.pkl']

In [870]:
from sklearn.pipeline import Pipeline

pipeline_svr = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("svr", SVR(kernel="rbf", gamma=0.01, coef0=0.6, C=0.9)),
    ]
)

pipeline_svr.fit(X_train, y_train)

Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scale', StandardScaler()),
                ('svr', SVR(C=0.9, coef0=0.6, gamma=0.01))])

In [871]:
y_pred = pipeline_svr.predict(X_test)

In [872]:
r2_score(y_test, y_pred)

0.6267205612460917

In [875]:
joblib.dump(pipeline_svr, "whole_pipeline_svr.pkl")

['whole_pipeline_svr.pkl']

In [877]:
pipeline_LinReg = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("lin_reg", LinearRegression()),
    ]
)

pipeline_LinReg.fit(X_train, y_train)

joblib.dump(pipeline_svr, "whole_pipeline_lin_reg.pkl")

['whole_pipeline_lin_reg.pkl']

In [879]:
new_df.to_csv("Cleaned_data.csv")